# Dependencies

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torch.utils.data import Subset

import numpy as np
import matplotlib.pyplot as plt
# import joblib

import os
# import zipfile
from pathlib import Path

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Current device: {device}")

# IC-U-Net

In [ ]:
class CBR_Block(nn.Module):
    """Convolutional, Batch norm, Relu activation function block x2

    """
    def __init__(self, in_channel, out_channel):
        super(CBR_Block, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channel, out_channel, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channel),
            nn.ReLU(),
            nn.Conv1d(out_channel, out_channel, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channel),
            nn.ReLU()
        )

    def forward(self, x):
        return self.conv(x)
    
    
class IC_UNet(nn.Module):
    """U-net model that downsamples, upsamples and concats to encode and decode EEG signals
    
    """
    def __init__(self, in_channel, out_channel):
        super(IC_UNet, self).__init__()
        
        # Encoder
        self.enc1 = CBR_Block(in_channel, out_channel)
        self.pool1 = nn.MaxPool1d(2)
        
        self.enc2 = CBR_Block(out_channel, out_channel * 2)
        self.pool2 = nn.MaxPool1d(2)
        
        self.enc3 = CBR_Block(out_channel * 2, out_channel * 4)
        self.pool3 = nn.MaxPool1d(2)
        
        self.enc4 = CBR_Block(out_channel * 4, out_channel * 8)
        
        # Decoder
        self.up4 = nn.ConvTranspose1d(out_channel * 8, out_channel * 4, kernel_size=2, stride=2)
        self.dec4 = CBR_Block(out_channel * 8, out_channel * 4)
        
        self.up3 = nn.ConvTranspose1d(out_channel * 4, out_channel * 2, kernel_size=2, stride=2)
        self.dec3 = CBR_Block(out_channel * 4, out_channel * 2)
        
        self.up2 = nn.ConvTranspose1d(out_channel * 2, out_channel, kernel_size=2, stride=2)
        self.dec2 = CBR_Block(out_channel * 2, out_channel)
        
        # Final conv maps decoder
        self.final_conv = nn.Conv1d(out_channel, in_channel, kernel_size=1)

    def forward(self, x):
        # Encoder
        d1 = self.enc1(x)
        p1 = self.pool1(d1)
        
        d2 = self.enc2(p1)
        p2 = self.pool2(d2)
        
        d3 = self.enc3(p2)
        p3 = self.pool3(d3)
        
        d4 = self.enc4(p3)
        
        # Decoder
        u4 = self.up4(d4)
        u4 = torch.cat([u4, d3], dim=1)
        u4 = self.dec4(u4)
        
        u3 = self.up3(u4)
        u3 = torch.cat([u3, d2], dim=1)
        u3 = self.dec3(u3)
        
        u2 = self.up2(u3)
        u2 = torch.cat([u2, d1], dim=1)
        u2 = self.dec2(u2)
                
        out = self.final_conv(u2)
        return out

# Dataset & Dataloader

In [ ]:
class EEG_Dataset(Dataset):
    """
    A PyTorch Dataset for loading raw and clean EEG epoch pairs.
    It expects the following structure:
    - data/
        - raw/
            - raw_training_epochs/
                - subject1/
                    - c3_epoch0_raw.pt
        - clean/
            - clean_training_epochs/
                - subject1/
                    - c3_epoch0_clean.pt
    """
    def __init__(self, data_dir, split, transform=None):
        """
        Args:
            data_dir (str): The path to the root 'data' directory.
            split (str): The dataset split (e.g., 'training_epochs', 'validation_epochs', or 'test_epochs').
        """
        self.transform = transform
        
        # Construct the paths to the raw and clean data folders for the specified split
        self.raw_dir = Path(data_dir) / 'raw' / f'raw_{split}'
        self.clean_dir = Path(data_dir) / 'clean' / f'clean_{split}'

        if not self.raw_dir.is_dir() or not self.clean_dir.is_dir():
            raise FileNotFoundError(f"One of the specified directories does not exist: {self.raw_dir} or {self.clean_dir}")

        self.file_pairs = []

        # Traverse the directory to find all raw files and their corresponding clean files
        for subject_dir in self.raw_dir.iterdir():
            if subject_dir.is_dir():
                for raw_file_path in subject_dir.glob('*.pt'):
                    # The clean file path is found by replacing the directory and file suffix
                    relative_path = raw_file_path.relative_to(self.raw_dir)
                    clean_file_path = self.clean_dir / relative_path.with_name(
                        raw_file_path.stem.replace('_raw', '_clean') + '.pt'
                    )

                    if clean_file_path.is_file():
                        self.file_pairs.append((raw_file_path, clean_file_path))
                    # else:
                    #     print(f"Warning: Corresponding clean file not found for {raw_file_path}")

    def __len__(self):
        """Returns the total number of data samples."""
        return len(self.file_pairs)

    def __getitem__(self, idx):
      """Loads and returns a raw and clean pair, normalized to [-1, 1]."""
      raw_path, clean_path = self.file_pairs[idx]

      # Load the tensors from their file paths
      raw_tensor = torch.load(raw_path)
      clean_tensor = torch.load(clean_path)

      # Add channel dim: [512] -> [1, 512]
      raw_tensor = raw_tensor.unsqueeze(0)
      clean_tensor = clean_tensor.unsqueeze(0)

      # Standardize each sample to [-1, 1]
      def normalize(tensor): # Comment out normalize function if already normalizing in data transformation
          min_val = tensor.min()
          max_val = tensor.max()
          if max_val > min_val:  # Avoid division by zero
              tensor = 2 * (tensor - min_val) / (max_val - min_val) - 1
          else:
              tensor = torch.zeros_like(tensor)
          return tensor

      raw_tensor = normalize(raw_tensor)
      
      if self.transform:
          raw_tensor = self.transform(raw_tensor)
          
      clean_tensor = normalize(clean_tensor)

      return raw_tensor, clean_tensor



# Dataset & Loaders examples

# ===== Training =====
train_dataset = EEG_Dataset(
    "/kaggle/input/eeg-clean-raw/dat-dataset-2-pro-max-supreme",
    "training_epochs"
)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)


# ===== Valid =====
valid_dataset = EEG_Dataset(
    "/kaggle/input/eeg-clean-raw/dat-dataset-2-pro-max-supreme",
    "validation_epochs"
)
valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=64, shuffle=False)


# ===== Test =====
test_dataset = EEG_Dataset(
    "/kaggle/input/eeg-clean-raw/dat-dataset-2-pro-max-supreme",
    "testing_epochs"
)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

# Training

In [ ]:
# ===== Hyperparameters =====
in_channel = 1
out_channel = 64
learning_rate = 1e-4


# ===== Model, Loss, Optimizer =====
model = IC_UNet(
    in_channel=in_channel,
    out_channel=out_channel
)
model = model.to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

num_epochs = 20
train_loss_history = []
val_loss_history = []

# ===== Paths for saving =====
checkpoint_dir = '/kaggle/working/models' # Change to desired file path
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(checkpoint_dir, 'icunet_checkpoint.pth')

# ===== Resume if model checkpoint exists =====
start_epoch = 0
train_loss_history = []
val_loss_history = []

if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    train_loss_history = checkpoint['train_loss_history']
    val_loss_history = checkpoint['val_loss_history']
    print(f"Resumed from epoch {start_epoch}")


# ===== Early Stopping Parameters =====
best_val_loss = float('inf')
epochs_no_improve = 0
best_model_state = None
patience = 20

# ===== Training =====
for epoch in range(start_epoch, num_epochs):
    model.train()
    train_running_loss = 0.0

    for batch_idx, (raw, clean) in enumerate(train_loader):
        raw = raw.to(device)
        clean = clean.to(device)

        optimizer.zero_grad()
        outputs = model(raw)
        loss = criterion(outputs, clean)
        loss.backward()
        optimizer.step()

        train_running_loss += loss.item()

    train_avg_loss = train_running_loss / len(train_loader)
    train_loss_history.append(train_avg_loss)

    # ====== Validation ======
    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for val_raw, val_clean in valid_loader:
            val_raw = val_raw.to(device)
            val_clean = val_clean.to(device)

            val_outputs = model(val_raw)
            val_loss = criterion(val_outputs, val_clean)
            val_running_loss += val_loss.item()

    val_avg_loss = val_running_loss / len(valid_loader)
    val_loss_history.append(val_avg_loss)

    # ===== Early Stopping Check =====
    if val_avg_loss < best_val_loss:
        best_val_loss = val_avg_loss
        epochs_no_improve = 0
        best_model_state = model.state_dict()
    else:
        epochs_no_improve += 1

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_avg_loss:.6f} | Val Loss: {val_avg_loss:.6f} | No Improve: {epochs_no_improve}")

    if epochs_no_improve >= patience:
        print(f"Early stopping after {epoch+1} epochs.")
        break
        
    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss_history': train_loss_history,
            'val_loss_history': val_loss_history,
            'hyperparameters': {
                'in_channel': in_channel,
                'out_channel': out_channel,
                'learning_rate': learning_rate,
                'num_epochs': num_epochs
            }
        }, checkpoint_path)
        print(f"Checkpoint saved to: {checkpoint_path}")



# ===== Saving model =====
model_dir = '/kaggle/working/models' # Change to desired file path
os.makedirs(model_dir, exist_ok=True)
model_cpu = model.to('cpu') # Switching back to cpu

# Save the model state dictionary
model_path = os.path.join(model_dir, 'icunet.pth')

# Save training history and hyperparameters
torch.save({
    'train_loss_history': train_loss_history,
    'val_loss_history': val_loss_history,
    'hyperparameters': {
        'in_channel': in_channel,
        'out_channel': out_channel,
        'learning_rate': learning_rate,
        'num_epochs': num_epochs
    }
}, model_path)
print(f"Training history saved to: {model_path}")

# Metrics

In [ ]:
# Metrics to measure success for regression task

def compute_snr(raw, clean):
    noise = raw - clean
    signal_power = np.mean(clean ** 2)
    noise_power = np.mean(noise ** 2)
    snr = 10 * np.log10(signal_power / noise_power)

    return snr

def compute_rrmse_time(raw, clean):
    numerator = np.sqrt(np.mean((raw - clean) ** 2))
    denominator = np.sqrt(np.mean(clean ** 2))
    return numerator / denominator

def compute_rrmse_freq(raw, clean):
    raw_fft = np.abs(np.fft.rfft(raw))
    clean_fft = np.abs(np.fft.rfft(clean))
    numerator = np.sqrt(np.mean((raw_fft - clean_fft) ** 2))
    denominator = np.sqrt(np.mean(clean_fft ** 2))
    return numerator / denominator

def compute_average_cc(raw, clean):
    correlations = []
    for i in range(raw.shape[0]):
        r = np.corrcoef(raw[i], clean[i])[0, 1]
        correlations.append(r)
    return np.mean(correlations)


def evaluate_model_metrics(model, dataloader):
    model.eval()
    snr_list = []
    rrmse_time_list = []
    rrmse_freq_list = []
    cc_list = []

    # Testing
    with torch.no_grad():
        for raw, clean in dataloader:
            # Run model forward
            model_output_batch = model(raw)

            # Move tensors to CPU and convert to numpy
            raw_np = raw.cpu().numpy()
            clean_np = clean.cpu().numpy()
            output_np = model_output_batch.cpu().numpy()

            # Compute metrics per sample in batch
            for i in range(raw_np.shape[0]):
                # raw_sample = raw_np[i]
                clean_sample = clean_np[i]
                model_output_sample = output_np[i]

                # Use model output and clean target (or raw if comparing raw to clean)
                snr_val = compute_snr(model_output_sample, clean_sample)
                rrmse_time_val = compute_rrmse_time(model_output_sample, clean_sample)
                rrmse_freq_val = compute_rrmse_freq(model_output_sample, clean_sample)
                cc_val = compute_average_cc(model_output_sample, clean_sample)

                snr_list.append(snr_val)
                rrmse_time_list.append(rrmse_time_val)
                rrmse_freq_list.append(rrmse_freq_val)
                cc_list.append(cc_val)

    # Aggregate metric results across entire dataset
    metrics = {
        "SNR": np.mean(snr_list),
        "RRMSE_time": np.mean(rrmse_time_list),
        "RRMSE_freq": np.mean(rrmse_freq_list),
        "Average_CC": np.mean(cc_list)
    }
    return metrics

# Testing

In [ ]:
# Printing metrics
metrics = evaluate_model_metrics(model, test_loader)

print("Test metrics:")
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")
    
with open('kaggle/working/metrics_output.txt', 'w') as file:
    for metric, value in metrics.items():
        file.write(f"{metrics}: {value}")